In [2]:
import json
import os
import time
import urllib.request
import uuid
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# O notebook roda a partir da pasta notebooks/; o .env fica na raiz do projeto
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ENV_FILE = PROJECT_ROOT / ".env"
if not ENV_FILE.is_file():
    raise FileNotFoundError(f"Arquivo .env não encontrado em {ENV_FILE}. Copie o .env.example para .env.")
load_dotenv(ENV_FILE, override=False)  # variáveis do container têm prioridade sobre o .env


def env(nome):
    """Lê uma variável obrigatória do ambiente; falha com mensagem clara se faltar."""
    valor = os.environ.get(nome, "").strip()
    if not valor:
        raise ValueError(f"Variável {nome} não definida no .env")
    return valor


# Resultado de cada teste: serviço, status e detalhe
resultados = []


def registrar(servico, ok, detalhe):
    resultados.append({"serviço": servico, "status": "OK" if ok else "FALHOU", "detalhe": detalhe})
    print(f"[{'OK' if ok else 'FALHOU'}] {servico}: {detalhe}")


def limpar(descricao, acao):
    """Apaga o que o teste criou. Se o serviço estiver fora do ar, avisa sem esconder a falha principal."""
    try:
        acao()
    except Exception as erro:
        print(f"[AVISO] limpeza de {descricao} não concluída: {type(erro).__name__}: {erro}")


print("Projeto:", PROJECT_ROOT)
print("Postgres:", env("POSTGRES_HOST"), "| MinIO:", env("S3_ENDPOINT_URL"),
      "| Kafka:", env("KAFKA_BOOTSTRAP_SERVERS"), "| Connect:", env("KAFKA_CONNECT_URL"))

Projeto: /app
Postgres: postgres | MinIO: http://minio:9000 | Kafka: kafka:29092 | Connect: http://connect:8083


In [10]:
import psycopg2


def conectar_postgres(usuario, senha):
    return psycopg2.connect(
        host=env("POSTGRES_HOST"), port=int(env("POSTGRES_PORT")), dbname=env("POSTGRES_DB"),
        user=usuario, password=senha, client_encoding="UTF8", application_name="notebook_teste",
    )


try:
    conn = conectar_postgres(env("POSTGRES_USER"), env("POSTGRES_PASSWORD"))
    with conn, conn.cursor() as cur:
        cur.execute("SELECT * FROM clientes LIMIT 10")
        clientes = cur.fetchall()

    conn.close()
    
    print("--- 10 Primeiros Clientes ---")
    for cliente in clientes:
        print(cliente)

except Exception as erro:
    registrar("PostgreSQL", False, f"{type(erro).__name__}: {erro}")
    

--- 10 Primeiros Clientes ---
('78d8cdd6-490a-486c-b644-29395256f928', 'Sr. Paulo Borges', 'maria-luiza31.19cb84c8@example.com', datetime.datetime(2025, 11, 10, 4, 57, 55, 914902, tzinfo=datetime.timezone.utc), datetime.datetime(2025, 11, 10, 4, 57, 55, 914902, tzinfo=datetime.timezone.utc))
('a011d446-20a1-45c8-8962-31dc4b67563a', 'Joana Costa', 'dante23.3a880cec@example.com', datetime.datetime(2025, 11, 10, 10, 27, 31, 199709, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 1, 27, 8, 49, 33, 920010, tzinfo=datetime.timezone.utc))
('9d36dea0-6619-4848-99bf-0f3b52ed2487', 'Yan Barros', 'lorena32.976b5158@example.com', datetime.datetime(2025, 11, 10, 13, 24, 18, 950673, tzinfo=datetime.timezone.utc), datetime.datetime(2025, 11, 10, 13, 24, 18, 950673, tzinfo=datetime.timezone.utc))
('0d46e159-a668-4e80-b16d-4fe8ac504475', 'Nicolas Cirino', 'laurafreitas.6ebb7a17@example.com', datetime.datetime(2025, 11, 10, 16, 22, 28, 513995, tzinfo=datetime.timezone.utc), datetime.datetime(2025